# MVA v4 · NB05A2 — BUB1B ClinVar control curation for functional-assay validation
**v1.0.2 · public-data-only · prospective assay-control audit · no PS3/BS3 assignment**

## Why this notebook exists

NB05A froze the F0 BUBR1 functional experiment and correctly capped the current functional-evidence framework at
`SUPPORTING_CANDIDATE_MAX`.

The unresolved question is now narrower:

> **Can we construct a sufficiently large, independently classified BUB1B pathogenic/likely-pathogenic and
> benign/likely-benign control panel to make a future ClinGen PS3/BS3 Moderate calibration realistically possible?**

This notebook does **not** treat a ClinVar label as automatically independent evidence. It:

1. downloads the current public ClinVar summary resources;
2. finds all BUB1B P/LP and B/LB candidates;
3. excludes aggregate conflicts/VUS and assay-incompatible consequence classes;
4. audits the legacy NB04B/NB05A comparator set against current ClinVar;
5. flags variants citing the key BUBR1 functional paper (PMID **20516114**) as a **circularity risk** for this assay family;
6. creates a manual evidence-independence review sheet;
7. computes both:
   - the **current confirmed control count**, and
   - a conservative **upper bound** if all high-priority candidates survive manual review.

### Critical rule

A variant counts toward ClinGen assay validation only after its P/LP or B/LB classification is shown to be
**independent of the functional assay being validated**.

Therefore this notebook can prove that a Moderate-capable panel is *feasible*, but it cannot by itself
upgrade PS3/BS3 strength.

ClinGen SVI guidance: without rigorous statistical calibration, at least **11 total pathogenic + benign
variant controls** are required to reach Moderate-level functional evidence.

No patient-level private data are used.


## v1.0.2 parser/schema patch

This version supersedes the failed v1.0.1 run.

Hardening:
- fixes the normalized-header/usecols mismatch (`#VariationID` on disk vs `VariationID` after normalization);
- detects the true ClinVar TSV header after metadata/preamble lines;
- skips the physical header row and supplies explicit normalized column names to pandas;
- chunk-parses `submission_summary` and filters to BUB1B VariationIDs;
- supports current `var_citations` citation-source/citation-id layout;
- keeps the explicit VariationID merge and 3-letter → 1-letter HGVS protein crosswalk;
- fails explicitly if a requested `usecols` field is absent;
- fixes the final output listing glob to `v1_0_2`;
- clears stale execution outputs so the next run is provenance-clean.

The biological design, ClinGen evidence gates, circularity rules and privacy boundary are unchanged from NB05A/NB05A2.


In [ ]:
import os
import io, re, json, gzip, hashlib, time
from pathlib import Path
from datetime import datetime, timezone, date
import numpy as np
import pandas as pd
import requests

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path(os.environ.get("MVA_ROOT", "/path/to/MVA_Hackathon_2026"))
except Exception:
    ROOT = Path(".")

PUBLIC = ROOT / "03_PUBLIC_ARTIFACTS"
PUBRES = ROOT / "05_PUBLIC_RESOURCES"
PUBLIC.mkdir(parents=True, exist_ok=True)
PUBRES.mkdir(parents=True, exist_ok=True)

SCHEMA = "MVA-NB05A2-BUB1B-CLINVAR-CONTROL-CURATION-1.0.2"
CREATED_UTC = datetime.now(timezone.utc).isoformat()

print("ROOT:", ROOT)
print("SCHEMA:", SCHEMA)


## 1. Require the frozen NB05A design

NB05A2 is a sidecar to NB05A. It must not change F0 endpoints or thresholds.


In [ ]:
REQ = {
    "handoff": PUBLIC / "NB05A_HANDOFF_v1_0_0.json",
    "prereg": PUBLIC / "NB05A_PREREGISTRATION_v1_0_0.json",
    "legacy_comparators": PUBLIC / "NB05A_EXISTING_BUBR1_COMPARATOR_AUDIT_v1_0_0.csv",
}

for k,p in REQ.items():
    if not p.exists():
        raise FileNotFoundError(f"Required NB05A artifact missing: {p}")

nb05a = json.loads(REQ["handoff"].read_text())
prereg = json.loads(REQ["prereg"].read_text())
legacy = pd.read_csv(REQ["legacy_comparators"])

assert nb05a["status"] == "DESIGN_COMPLETE_AWAITING_EXPERIMENT"
assert nb05a["ps3_bs3"]["assigned"] is False
assert nb05a["ps3_bs3"]["current_assay_validation_cap"] == "SUPPORTING_CANDIDATE_MAX"
assert prereg["target"]["gene"] == "BUB1B"

print(json.dumps(nb05a, indent=2))
display(legacy)


## 2. Freeze public sources and circularity rule

The current BUBR1 functional assay family is anchored to Suijkerbuijk et al. 2010 (PMID **20516114**).
A candidate whose public classification cites that paper is **not automatically invalid**, but it cannot be
counted as an independent validation control until a curator demonstrates that the clinical classification does
not depend on the assay family being calibrated.

This intentionally favors false negatives over circular validation.


In [ ]:
URLS = {
    "variant_summary": "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz",
    "submission_summary": "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/submission_summary.txt.gz",
    "var_citations": "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/var_citations.txt",
}

CIRCULARITY_PMIDS = {"20516114"}  # same key BUBR1 functional paper used to motivate the assay family
TARGET_PROTEIN = "N1002K"
TARGET_HGVS = "NM_001211.6:c.3006T>G"

run_day = datetime.now(timezone.utc).date().isoformat()
CACHE = PUBRES / "clinvar" / run_day
CACHE.mkdir(parents=True, exist_ok=True)

def download(url, path, min_bytes=100):
    path = Path(path)
    if path.exists() and path.stat().st_size >= min_bytes:
        return path
    r = requests.get(url, timeout=180)
    r.raise_for_status()
    tmp = path.with_suffix(path.suffix + ".part")
    tmp.write_bytes(r.content)
    tmp.replace(path)
    return path

paths = {
    "variant_summary": download(URLS["variant_summary"], CACHE/"variant_summary.txt.gz", 100000),
    "submission_summary": download(URLS["submission_summary"], CACHE/"submission_summary.txt.gz", 100000),
    "var_citations": download(URLS["var_citations"], CACHE/"var_citations.txt", 1000),
}

source_audit = []
for key,p in paths.items():
    source_audit.append({
        "resource": key,
        "path": str(p),
        "bytes": p.stat().st_size,
        "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
        "url": URLS[key],
    })
source_audit = pd.DataFrame(source_audit)
display(source_audit)
source_audit.to_csv(PUBLIC/"NB05A2_CLINVAR_SOURCE_AUDIT_v1_0_2.csv", index=False)


## 3. Read ClinVar robustly and isolate BUB1B

The notebook uses GRCh38 rows for unique variant-level curation. It does not use the patient's VCF.


In [ ]:
import csv

def _open_text_auto(path):
    path = Path(path)
    if path.suffix == ".gz":
        return gzip.open(path, "rt", encoding="utf-8", errors="replace", newline="")
    return open(path, "rt", encoding="utf-8", errors="replace", newline="")

def _norm_header_name(x):
    return str(x).strip().lstrip("#").strip()

def detect_tsv_header(path, required_columns, max_scan_lines=500):
    """Detect the actual TSV header after optional ClinVar metadata/preamble lines."""
    req = {str(x).lower() for x in required_columns}
    with _open_text_auto(path) as fh:
        for i, line in enumerate(fh):
            if i >= max_scan_lines:
                break
            if "\t" not in line:
                continue
            names = [_norm_header_name(x) for x in line.rstrip("\n\r").split("\t")]
            low = {x.lower() for x in names}
            if req.issubset(low):
                if len(names) != len(set(names)):
                    raise RuntimeError(f"Duplicate normalized columns in {path}: {names}")
                return i, names
    raise RuntimeError(
        f"Could not detect TSV header in {path}; required columns={sorted(required_columns)}"
    )

def read_clinvar_tsv(path, required_columns, usecols=None, chunksize=None):
    """Read ClinVar TSV safely when metadata/preamble may precede the real header."""
    header_line, header_names = detect_tsv_header(path, required_columns)
    if usecols is not None:
        wanted = {_norm_header_name(x).lower() for x in usecols}
        actual_usecols = [c for c in header_names if c.lower() in wanted]
        missing_usecols = sorted(wanted - {c.lower() for c in actual_usecols})
        if missing_usecols:
            raise RuntimeError(
                f"Requested columns not found in detected header for {path}: {missing_usecols}"
            )
    else:
        actual_usecols = None

    # ClinVar headers can physically begin with '#', e.g. '#VariationID'.
    # The detector has already normalized the header names. Skip the physical
    # header row and supply those normalized names explicitly so `usecols`
    # is matched against the same schema used downstream.
    reader = pd.read_csv(
        path,
        sep="\t",
        compression="infer",
        dtype=str,
        low_memory=False,
        skiprows=header_line + 1,
        header=None,
        names=header_names,
        usecols=actual_usecols,
        chunksize=chunksize,
        encoding="utf-8",
        encoding_errors="replace",
        on_bad_lines="error",
    )

    def normalize(df):
        df = df.copy()
        df.columns = [_norm_header_name(c) for c in df.columns]
        return df

    if chunksize is None:
        return normalize(reader), header_line, header_names

    def gen():
        for chunk in reader:
            yield normalize(chunk)
    return gen(), header_line, header_names

vs, vs_header_line, vs_header_names = read_clinvar_tsv(
    paths["variant_summary"],
    required_columns=["GeneSymbol", "Assembly", "VariationID", "ClinicalSignificance"],
)
vc, vc_header_line, vc_header_names = read_clinvar_tsv(
    paths["var_citations"],
    required_columns=["VariationID"],
)

# submission_summary is large and may have metadata lines before its TSV header.
# Detect its header now; stream/filter only BUB1B rows after BUB1B VariationIDs are known.
ss_header_line, ss_header_names = detect_tsv_header(
    paths["submission_summary"],
    required_columns=["VariationID", "ClinicalSignificance"],
)
ss = None

print("variant_summary header line:", vs_header_line + 1)
print("submission_summary header line:", ss_header_line + 1)
print("var_citations header line:", vc_header_line + 1)
print("variant_summary columns:", list(vs.columns))
print("submission_summary columns:", ss_header_names)
print("var_citations columns:", list(vc.columns))

def col(df_or_columns, *candidates):
    cols = list(df_or_columns.columns) if hasattr(df_or_columns, "columns") else list(df_or_columns)
    norm = {_norm_header_name(c).lower(): c for c in cols}
    for cand in candidates:
        hit = norm.get(str(cand).lower())
        if hit is not None:
            return hit
    return None

gene_c = col(vs, "GeneSymbol")
asm_c = col(vs, "Assembly")
varid_c = col(vs, "VariationID")
name_c = col(vs, "Name")
clinsig_c = col(vs, "ClinicalSignificance")
review_c = col(vs, "ReviewStatus")
nsub_c = col(vs, "NumberSubmitters")
phen_c = col(vs, "PhenotypeList")

required = [gene_c, asm_c, varid_c, name_c, clinsig_c, review_c]
if any(x is None for x in required):
    raise RuntimeError("ClinVar variant_summary schema changed; required columns not found.")

vs[varid_c] = vs[varid_c].astype(str)
bub = vs[(vs[gene_c] == "BUB1B") & (vs[asm_c] == "GRCh38")].copy()
bub = bub.drop_duplicates(subset=[varid_c]).reset_index(drop=True)
print("Unique BUB1B GRCh38 VariationIDs:", len(bub))


## 4. Conservative aggregate classification and assay compatibility

Only clean aggregate P/LP or B/LB classes enter the candidate pool.

The following are excluded before any manual review:
- VUS;
- conflicting classifications;
- risk-factor/association-only states;
- N1002K itself;
- variants that cannot be meaningfully modeled in the same protein-function/complementation framework.

Splice-only and synonymous controls are retained in the full audit but not in the primary protein-function pool.


In [ ]:
PLP_EXACT = {
    "pathogenic", "likely pathogenic", "pathogenic/likely pathogenic",
    "likely pathogenic/pathogenic"
}
BLB_EXACT = {
    "benign", "likely benign", "benign/likely benign", "likely benign/benign"
}

def clean_class(x):
    s = str(x).strip().lower()
    if "conflict" in s or "uncertain" in s:
        return "EXCLUDE_UNCERTAIN_OR_CONFLICT"
    if s in PLP_EXACT:
        return "P_LP"
    if s in BLB_EXACT:
        return "B_LB"
    return "EXCLUDE_OTHER"

def review_rank(x):
    s = str(x).lower()
    if "practice guideline" in s:
        return 4
    if "expert panel" in s:
        return 3
    if "multiple submitters" in s and "no conflicts" in s:
        return 2
    if "criteria provided" in s and "single submitter" in s:
        return 1
    return 0

def consequence_from_name(name):
    s = str(name)
    low = s.lower()
    if "(p." in s:
        p = s.split("(p.",1)[1].split(")",1)[0]
        if "=" in p:
            return "synonymous"
        if "fs" in p.lower():
            return "frameshift"
        if "ter" in p.lower() or "*" in p:
            return "nonsense"
        if "delins" in p.lower():
            return "inframe_delins"
        if "del" in p.lower() or "dup" in p.lower():
            return "inframe_indel"
        return "missense_or_other_protein_change"
    # canonical splice heuristic from c. notation
    m = re.search(r"c\.\d+([+-])(\d+)", s)
    if m:
        d = int(m.group(2))
        return "canonical_splice" if d <= 2 else "intronic_or_splice_region"
    return "non_protein_or_unresolved"

def protein_change(name):
    m = re.search(r"\(p\.([^\)]+)\)", str(name))
    return m.group(1) if m else ""

bub["aggregate_control_class"] = bub[clinsig_c].map(clean_class)
bub["review_rank"] = bub[review_c].map(review_rank)
bub["consequence_class"] = bub[name_c].map(consequence_from_name)
bub["protein_change"] = bub[name_c].map(protein_change)
bub["n_submitters"] = pd.to_numeric(bub[nsub_c], errors="coerce").fillna(0).astype(int) if nsub_c else 0

protein_compatible = {
    "missense_or_other_protein_change",
    "nonsense",
    "frameshift",
    "inframe_delins",
    "inframe_indel",
}
bub["protein_function_assay_compatible"] = bub["consequence_class"].isin(protein_compatible)
bub["is_target_N1002K"] = bub[name_c].astype(str).str.contains(
    r"Asn1002Lys|N1002K|c\.3006T>G", regex=True, na=False
)

display(
    bub["aggregate_control_class"].value_counts(dropna=False)
    .rename_axis("class").reset_index(name="n")
)


## 5. Citation-level circularity audit

`var_citations.txt` is used as a conservative screen. A citation to PMID 20516114 means
**manual independence review is mandatory**.

Absence of that PMID does **not** prove independence.


In [ ]:
vc_varid = col(vc, "VariationID")
if vc_varid is None:
    raise RuntimeError(f"VariationID not found in var_citations columns: {list(vc.columns)}")

# Current ClinVar var_citations commonly represents PubMed as citation_source + citation_id.
# Direct PMID/PubMedID columns remain supported for forward/backward compatibility.
vc_pmid = col(vc, "PMID", "PubMedID")
vc_source = col(vc, "citation_source", "CitationSource", "Source")
vc_citation_id = col(vc, "citation_id", "CitationID")

vc2 = vc.copy()
vc2[vc_varid] = vc2[vc_varid].astype(str)

if vc_pmid is not None:
    vc_pubmed = vc2[[vc_varid, vc_pmid]].rename(columns={vc_pmid: "pmid"}).copy()
elif vc_source is not None and vc_citation_id is not None:
    src_norm = vc2[vc_source].astype(str).str.strip().str.lower()
    is_pubmed = src_norm.isin({"pubmed", "pmid", "pubmed/medline"})
    vc_pubmed = vc2.loc[is_pubmed, [vc_varid, vc_citation_id]].rename(
        columns={vc_citation_id: "pmid"}
    ).copy()
else:
    raise RuntimeError(
        "Unsupported var_citations schema for PubMed extraction. "
        f"Columns: {list(vc.columns)}"
    )

vc_pubmed["pmid"] = (
    vc_pubmed["pmid"].astype(str)
    .str.replace(r"^PMID:", "", regex=True)
    .str.strip()
)
vc_pubmed = vc_pubmed[
    vc_pubmed["pmid"].ne("") & vc_pubmed["pmid"].ne("nan")
].copy()

pmid_map = (
    vc_pubmed.groupby(vc_varid)["pmid"]
    .apply(lambda s: sorted({str(x) for x in s if str(x) and str(x) != "nan"}))
    .to_dict()
)

bub["public_pmids"] = bub[varid_c].astype(str).map(
    lambda v: ";".join(pmid_map.get(v, []))
)
bub["same_assay_publication_flag"] = bub["public_pmids"].map(
    lambda x: bool(CIRCULARITY_PMIDS.intersection(set(filter(None, str(x).split(";")))))
)

print("BUB1B variants carrying circularity PMID(s):",
      int(bub["same_assay_publication_flag"].sum()))


## 6. Submission-level concordance audit

Aggregate classifications are not enough. For each candidate we summarize the current SCV-level
classifications available in `submission_summary.txt.gz`.

This is still a screening layer, not the final evidence-independence decision.


In [ ]:
ss_varid = col(ss_header_names, "VariationID")
ss_sig = col(ss_header_names, "ClinicalSignificance")
ss_review = col(ss_header_names, "ReviewStatus")
ss_submitter = col(ss_header_names, "Submitter")
ss_scv = col(ss_header_names, "SCV", "SCVAccession", "ClinVarAccession")
ss_desc = col(ss_header_names, "Description", "Explanation")

if ss_varid is None or ss_sig is None:
    raise RuntimeError(
        "ClinVar submission_summary schema changed; VariationID/ClinicalSignificance not found. "
        f"Columns: {ss_header_names}"
    )

wanted_ss = [x for x in [ss_varid, ss_sig, ss_review, ss_submitter, ss_scv, ss_desc] if x]
target_ids = set(bub[varid_c].astype(str))

ss_chunks, _, _ = read_clinvar_tsv(
    paths["submission_summary"],
    required_columns=["VariationID", "ClinicalSignificance"],
    usecols=wanted_ss,
    chunksize=250_000,
)

frames = []
n_ss_rows_scanned = 0
for chunk in ss_chunks:
    chunk_varid = col(chunk, "VariationID")
    if chunk_varid is None:
        raise RuntimeError("VariationID disappeared after chunked read of submission_summary.")
    n_ss_rows_scanned += len(chunk)
    hit = chunk[chunk[chunk_varid].astype(str).isin(target_ids)].copy()
    if len(hit):
        frames.append(hit)

if frames:
    ss_bub = pd.concat(frames, ignore_index=True)
else:
    ss_bub = pd.DataFrame(columns=[_norm_header_name(x) for x in wanted_ss])

ss_varid = col(ss_bub, "VariationID")
ss_sig = col(ss_bub, "ClinicalSignificance")
ss_review = col(ss_bub, "ReviewStatus")
ss_submitter = col(ss_bub, "Submitter")
ss_scv = col(ss_bub, "SCV", "SCVAccession", "ClinVarAccession")
ss_desc = col(ss_bub, "Description", "Explanation")

print("submission_summary rows scanned:", n_ss_rows_scanned)
print("BUB1B submission rows retained:", len(ss_bub))

if ss_varid is None or ss_sig is None:
    raise RuntimeError("Filtered submission_summary lacks required columns.")

ss_bub[ss_varid] = ss_bub[ss_varid].astype(str)
ss_bub["scv_control_class"] = ss_bub[ss_sig].map(clean_class)

def joinuniq(s):
    return ";".join(sorted({str(x) for x in s if pd.notna(x) and str(x) != "nan"}))

agg_dict = {ss_sig: joinuniq, "scv_control_class": joinuniq}
if ss_submitter: agg_dict[ss_submitter] = joinuniq
if ss_scv: agg_dict[ss_scv] = joinuniq
if ss_review: agg_dict[ss_review] = joinuniq
if ss_desc: agg_dict[ss_desc] = joinuniq

scv_summary = ss_bub.groupby(ss_varid, dropna=False).agg(agg_dict).reset_index()
rename = {ss_varid:"VariationID_scv", ss_sig:"SCV_significance"}
if ss_submitter: rename[ss_submitter] = "SCV_submitters"
if ss_scv: rename[ss_scv] = "SCV_accessions"
if ss_review: rename[ss_review] = "SCV_review_statuses"
if ss_desc: rename[ss_desc] = "SCV_descriptions"
scv_summary = scv_summary.rename(columns=rename)
scv_summary["VariationID_scv"] = scv_summary["VariationID_scv"].astype(str)

bub[varid_c] = bub[varid_c].astype(str)
bub = bub.merge(
    scv_summary,
    left_on=varid_c,
    right_on="VariationID_scv",
    how="left",
    validate="one_to_one",
)


## 7. Build the strict public candidate pool

`strict_public_candidate=True` means **worth manual independence curation**.

It does **not** mean `counts_for_clingen_validation=True`.

Primary strict screen:
- clean P/LP or B/LB aggregate;
- criteria provided by ≥2 concordant submitters / expert panel / practice guideline;
- protein-function assay compatible;
- not N1002K;
- no direct same-assay-publication flag.

A second table preserves candidates excluded only by the PMID circularity flag so they can be reviewed
rather than silently discarded.


In [ ]:
bub["strict_public_candidate"] = (
    bub["aggregate_control_class"].isin(["P_LP","B_LB"])
    & (bub["review_rank"] >= 2)
    & bub["protein_function_assay_compatible"]
    & (~bub["is_target_N1002K"])
    & (~bub["same_assay_publication_flag"])
)

bub["circularity_review_candidate"] = (
    bub["aggregate_control_class"].isin(["P_LP","B_LB"])
    & (bub["review_rank"] >= 1)
    & bub["protein_function_assay_compatible"]
    & (~bub["is_target_N1002K"])
    & bub["same_assay_publication_flag"]
)

keep_cols = [
    varid_c, name_c, clinsig_c, review_c, "n_submitters",
    "aggregate_control_class", "review_rank", "consequence_class", "protein_change",
    "public_pmids", "same_assay_publication_flag",
    "strict_public_candidate", "circularity_review_candidate"
]
if phen_c: keep_cols.insert(4, phen_c)

audit = bub[keep_cols].copy()
audit = audit.rename(columns={
    varid_c:"VariationID",
    name_c:"ClinVar_name",
    clinsig_c:"ClinicalSignificance",
    review_c:"ReviewStatus",
    phen_c:"PhenotypeList" if phen_c else phen_c,
})
audit.to_csv(PUBLIC/"NB05A2_BUB1B_CLINVAR_FULL_AUDIT_v1_0_2.csv", index=False)

strict = audit[audit["strict_public_candidate"]].copy()
circ = audit[audit["circularity_review_candidate"]].copy()

print("Strict public candidates:", len(strict))
print(strict["aggregate_control_class"].value_counts(dropna=False))
print("Circularity-review candidates:", len(circ))
display(strict.head(50))


## 8. Audit the legacy BUBR1 comparator panel against current ClinVar

This is deliberately diagnostic. Several classic MVA substitutions are expected to remain unsuitable as
validation controls because their current ClinVar state is VUS/conflicting or because the same functional paper
contributes to their interpretation.


In [ ]:
AA3_TO_1 = {
    "Ala":"A","Arg":"R","Asn":"N","Asp":"D","Cys":"C","Gln":"Q","Glu":"E","Gly":"G",
    "His":"H","Ile":"I","Leu":"L","Lys":"K","Met":"M","Phe":"F","Pro":"P","Ser":"S",
    "Thr":"T","Trp":"W","Tyr":"Y","Val":"V","Ter":"*",
}

def canonical_short_p(p):
    s = str(p).strip().replace("p.", "").strip("() ")
    s = s.replace("Ter", "*")
    if re.fullmatch(r"[A-Z*]\d+[A-Z*=]", s):
        return s
    m = re.fullmatch(r"([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2}|\*)", s)
    if m:
        a1, pos, a2 = m.groups()
        if a1 in AA3_TO_1:
            out2 = "*" if a2 == "*" else AA3_TO_1.get(a2)
            if out2 is not None:
                return f"{AA3_TO_1[a1]}{pos}{out2}"
    m = re.fullmatch(r"([A-Z][a-z]{2})(\d+)=", s)
    if m and m.group(1) in AA3_TO_1:
        return f"{AA3_TO_1[m.group(1)]}{m.group(2)}="
    return s

audit["protein_short"] = audit["protein_change"].map(canonical_short_p)
legacy_x = legacy.copy()
legacy_x["protein_short"] = legacy_x["variant"].astype(str).map(canonical_short_p)
legacy_cross = legacy_x.merge(
    audit,
    on="protein_short",
    how="left",
    suffixes=("_NB05A","_ClinVar"),
    validate="one_to_many",
)
legacy_cross.to_csv(PUBLIC/"NB05A2_LEGACY_COMPARATOR_CLINVAR_CROSSWALK_v1_0_2.csv", index=False)

print("Legacy variants matched to >=1 ClinVar row:",
      legacy_cross.groupby("variant")["VariationID"].apply(lambda s: s.notna().any()).sum(),
      "/", legacy["variant"].nunique())
display(legacy_cross)


## 9. Manual independence-review template

A candidate is **not counted** until a curator confirms that its clinical classification is independent of
the assay family being calibrated.

Recommended evidence:
- ClinVar SCV/VCV classification rationale;
- segregation/case evidence;
- population-frequency evidence;
- PVS1/LoF evidence where applicable;
- expert-panel or multi-submitter consensus;
- explicit check that PMID 20516114 / equivalent checkpoint-rescue data are not necessary to reach the
  control classification.

For benign controls, population/segregation evidence is especially useful because it can be independent of
the functional assay.

For pathogenic controls, avoid relying only on the same BUBR1 checkpoint assay that is being validated.


In [ ]:
manual = pd.concat([
    strict.assign(candidate_source="STRICT_PUBLIC_POOL"),
    circ.assign(candidate_source="CIRCULARITY_REVIEW_POOL"),
], ignore_index=True)

manual = manual[[
    "VariationID","ClinVar_name","ClinicalSignificance","ReviewStatus",
    "aggregate_control_class","consequence_class","protein_change","public_pmids",
    "same_assay_publication_flag","candidate_source"
]].copy()

manual["manual_independence_status"] = "PENDING"
manual["independent_evidence_basis"] = ""
manual["classification_source_checked"] = ""
manual["classification_date_checked"] = ""
manual["include_as_clingen_validation_control"] = False
manual["manual_exclusion_reason"] = ""
manual["curator_notes"] = ""

manual_path = PUBLIC/"NB05A2_MANUAL_INDEPENDENCE_REVIEW_TEMPLATE_v1_0_2.csv"
manual.to_csv(manual_path, index=False)
display(manual.head(50))
print("Template:", manual_path)


## 10. Feasibility upper bound and completed-review ingestion

Two quantities must never be confused:

- **confirmed_n** — manually verified independent controls; this is the only count relevant to an actual
  ClinGen validation claim.
- **strict_upper_bound_n** — high-priority public candidates before manual independence review.

If a curator later creates:

`03_PUBLIC_ARTIFACTS/NB05A2_MANUAL_INDEPENDENCE_REVIEW_COMPLETED_v1_0_2.csv`

this notebook will ingest it and recompute the confirmed panel.


In [ ]:
completed_path = PUBLIC/"NB05A2_MANUAL_INDEPENDENCE_REVIEW_COMPLETED_v1_0_2.csv"

if completed_path.exists():
    completed = pd.read_csv(completed_path)
    required_manual = {
        "VariationID","aggregate_control_class","manual_independence_status",
        "include_as_clingen_validation_control"
    }
    if not required_manual.issubset(completed.columns):
        raise RuntimeError(f"Completed manual review missing columns: {required_manual - set(completed.columns)}")
    completed["include_as_clingen_validation_control"] = (
        completed["include_as_clingen_validation_control"]
        .astype(str).str.lower().isin(["true","1","yes","y"])
    )
    confirmed = completed[
        (completed["manual_independence_status"].astype(str).str.upper() == "INDEPENDENT_CONFIRMED")
        & completed["include_as_clingen_validation_control"]
        & completed["aggregate_control_class"].isin(["P_LP","B_LB"])
    ].copy()
else:
    completed = None
    confirmed = pd.DataFrame(columns=manual.columns)

def counts(df):
    if len(df) == 0:
        return {"P_LP":0, "B_LB":0, "total":0}
    vc = df["aggregate_control_class"].value_counts()
    return {
        "P_LP": int(vc.get("P_LP",0)),
        "B_LB": int(vc.get("B_LB",0)),
        "total": int(len(df)),
    }

strict_counts = counts(strict)
confirmed_counts = counts(confirmed)

def mixed_moderate_possible(c):
    return c["total"] >= 11 and c["P_LP"] > 0 and c["B_LB"] > 0

if mixed_moderate_possible(confirmed_counts):
    panel_status = "INDEPENDENT_CONTROL_PANEL_NUMERIC_GATE_MET"
elif mixed_moderate_possible(strict_counts):
    panel_status = "MANUAL_CURATION_REQUIRED_POTENTIALLY_MODERATE_CAPABLE"
else:
    panel_status = "STRICT_PUBLIC_POOL_INSUFFICIENT_FOR_11_CONTROL_MODERATE_GATE"

summary = pd.DataFrame([
    {"level":"confirmed_independent", **confirmed_counts,
     "mixed_11_control_gate": mixed_moderate_possible(confirmed_counts)},
    {"level":"strict_public_upper_bound", **strict_counts,
     "mixed_11_control_gate": mixed_moderate_possible(strict_counts)},
])
display(summary)
print("PANEL STATUS:", panel_status)


## 11. Consequence-balance safeguard

A numerically sufficient panel can still be biologically weak if all pathogenic controls are truncating
while all benign controls are missense.

This is not a formal ClinGen rule, but NB05A2 reports the imbalance because it can inflate apparent
discrimination for an assay that is supposed to interpret a missense target.


In [ ]:
if len(strict):
    balance = pd.crosstab(
        strict["aggregate_control_class"],
        strict["consequence_class"],
        margins=True
    )
else:
    balance = pd.DataFrame()
display(balance)

strict_missense_plp = int(((strict["aggregate_control_class"]=="P_LP") &
                           (strict["consequence_class"]=="missense_or_other_protein_change")).sum())
strict_missense_blb = int(((strict["aggregate_control_class"]=="B_LB") &
                           (strict["consequence_class"]=="missense_or_other_protein_change")).sum())

balance_note = {
    "strict_missense_P_LP": strict_missense_plp,
    "strict_missense_B_LB": strict_missense_blb,
    "warning": (
        "A >=11 mixed panel is only a numeric ClinGen gate. "
        "Consequence/mechanism balance must still be reviewed before assay validation."
    )
}
print(json.dumps(balance_note, indent=2))


## 12. Current-era sanity checks — warnings only

These are not hard-coded classifications. They alert us if the current ClinVar release differs from the
public state observed when NB05A2 was authored.

Expected at authoring time:
- **R814H** — conflicting P/LP vs LB → not a clean control.
- **L1012P** — VUS → not a control.
- **L844F** — VUS → not a control.
- **Q921H** — conflicting VUS/LB and same functional-paper concern → not a clean control.
- **R550Q** — B/LB aggregate, but must undergo circularity/independence review before counting.
- **V4M** — current Likely benign aggregate; prospective benign-control candidate subject to independence review.


In [ ]:
expected = {
    "R814H": ("EXCLUDE_UNCERTAIN_OR_CONFLICT", None),
    "L1012P": ("EXCLUDE_UNCERTAIN_OR_CONFLICT", None),
    "L844F": ("EXCLUDE_UNCERTAIN_OR_CONFLICT", None),
    "Q921H": ("EXCLUDE_UNCERTAIN_OR_CONFLICT", None),
    "R550Q": ("B_LB", None),
    "V4M": ("B_LB", None),
}

rows=[]
for p,(exp,_) in expected.items():
    hit = audit[audit["protein_short"] == p]
    if len(hit)==0:
        rows.append({"variant":p,"observed":"NOT_FOUND","expected_at_authoring":exp,"changed":True})
    else:
        obs = ";".join(hit["aggregate_control_class"].astype(str).unique())
        rows.append({"variant":p,"observed":obs,"expected_at_authoring":exp,"changed":obs!=exp})
sanity = pd.DataFrame(rows)
display(sanity)
sanity.to_csv(PUBLIC/"NB05A2_CURRENT_CLINVAR_SANITY_CHECK_v1_0_2.csv", index=False)


## 13. Final handoff

Even if the numeric 11-control gate is met after manual curation, the actual functional-evidence strength
does not increase until those controls are tested in the **same assay instance** with adequate basic controls,
replication and performance.

Therefore the possible outcomes are:

- `STRICT_PUBLIC_POOL_INSUFFICIENT...` → Moderate calibration is not worth building from current public BUB1B controls.
- `MANUAL_CURATION_REQUIRED_POTENTIALLY_MODERATE_CAPABLE` → curate independence before selecting wet-lab validation controls.
- `INDEPENDENT_CONTROL_PANEL_NUMERIC_GATE_MET` → control selection can advance, but PS3/BS3 is still not assigned.


In [ ]:
HANDOFF = {
    "schema": SCHEMA,
    "created_utc": CREATED_UTC,
    "status": panel_status,
    "source_release_date": run_day,
    "source_sha256": {
        row["resource"]: row["sha256"] for _,row in source_audit.iterrows()
    },
    "nb05a_status": nb05a["status"],
    "current_ps3_bs3_assigned": False,
    "current_assay_validation_cap": nb05a["ps3_bs3"]["current_assay_validation_cap"],
    "confirmed_independent_controls": confirmed_counts,
    "strict_public_upper_bound": strict_counts,
    "strict_missense_balance": {
        "P_LP": strict_missense_plp,
        "B_LB": strict_missense_blb,
    },
    "moderate_numeric_gate_definition": ">=11 total independent mixed P/LP + B/LB controls",
    "moderate_numeric_gate_met_confirmed": mixed_moderate_possible(confirmed_counts),
    "moderate_numeric_gate_possible_upper_bound": mixed_moderate_possible(strict_counts),
    "same_assay_circularity_pmids": sorted(CIRCULARITY_PMIDS),
    "manual_review_file": str(manual_path),
    "next_step": (
        "manual evidence-independence curation of high-priority BUB1B controls"
        if panel_status.startswith("MANUAL_CURATION")
        else (
            "select and experimentally validate the confirmed control panel in the same assay instance"
            if panel_status.startswith("INDEPENDENT_CONTROL")
            else
            "retain Supporting cap or pursue formal statistical calibration / broader defensible control strategy"
        )
    ),
    "drug_selection": False,
    "privacy": {
        "private_patient_data_used": False,
        "patient_vcf_reopened": False,
        "public_sources_only": True,
    }
}

handoff_path = PUBLIC/"NB05A2_HANDOFF_v1_0_2.json"
handoff_path.write_text(json.dumps(HANDOFF, indent=2))

strict.to_csv(PUBLIC/"NB05A2_STRICT_PUBLIC_CONTROL_CANDIDATES_v1_0_2.csv", index=False)
circ.to_csv(PUBLIC/"NB05A2_CIRCULARITY_REVIEW_CANDIDATES_v1_0_2.csv", index=False)
summary.to_csv(PUBLIC/"NB05A2_CONTROL_COUNT_SUMMARY_v1_0_2.csv", index=False)

print(json.dumps(HANDOFF, indent=2))
print("\nWrote:")
for p in sorted(PUBLIC.glob("NB05A2_*v1_0_2*")):
    print(" -", p)


## Interpretation rule

NB05A2 is intentionally conservative.

A negative result — too few independent controls — is useful. It prevents us from claiming a
`PS3_Moderate` path that BUB1B cannot currently support without a different validation strategy.

A positive feasibility result is **not PS3 evidence**. It means only that a sufficiently sized candidate
validation panel exists and can be taken forward to manual curation and then wet-lab assay validation.
